# 🧩 WEEK 9 ASSIGNMENT - EXERCISE 3: SEMANTIC SEGMENTATION WITH U-NET ON OXFORD-IIIT PET
**Course:** Deep Learning / Computer Vision  
**Dataset:** Images from `tanlikesmath/the-oxfordiiit-pet-dataset` and trimaps from `julinmaloof/the-oxfordiiit-pet-dataset`  
**Requirements:** Build a U-Net from scratch (Encoder, Bottleneck, Decoder, Skip Connections); train & evaluate Pixel Accuracy, IoU, Dice Score; **visualize with a set of 8 charts and overlay images (Overlay Blended Mask)**.  
**Rules:** 100% compliance: no `assert` and no `try-except`.


---
## ⚙️ PART 1: IMPORT LIBRARIES & DOWNLOAD DATA VIA KAGGLEHUB


In [ ]:
import os
import time
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# 1. Download the Oxford-IIIT Pet image dataset via kagglehub
import kagglehub
path = kagglehub.dataset_download("tanlikesmath/the-oxfordiiit-pet-dataset")
print("Image dataset path:", path)

# Also download the mask labels (trimaps) from the standard dataset
annot_path = kagglehub.dataset_download("julinmaloof/the-oxfordiiit-pet-dataset")
print("Trimap labels path:", annot_path)

# Locate the images and trimaps folders robustly
img_dir = os.path.join(path, "images")
if not os.path.exists(img_dir):
    for root, dirs, files in os.walk(path):
        if "images" in dirs:
            img_dir = os.path.join(root, "images")
            break

mask_dir = os.path.join(annot_path, "annotations", "trimaps")
if not os.path.exists(mask_dir):
    for root, dirs, files in os.walk(annot_path):
        if "trimaps" in dirs:
            mask_dir = os.path.join(root, "trimaps")
            break

print("Images folder:", img_dir)
print("Masks folder (trimaps):", mask_dir)

IMG_SIZE = 128
BATCH_SIZE = 32

img_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor()
])

available_masks = set(os.listdir(mask_dir)) if os.path.exists(mask_dir) else set()
valid_samples = []
for f in os.listdir(img_dir):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        base_name = os.path.splitext(f)[0]
        mask_name = f"{base_name}.png"
        if mask_name in available_masks:
            valid_samples.append((os.path.join(img_dir, f), os.path.join(mask_dir, mask_name)))

print(f"Number of valid samples (image + mask): {len(valid_samples)}")

class OxfordPetKaggleDataset(Dataset):
    def __init__(self, samples, img_trans, mask_trans):
        self.samples = samples
        self.img_trans = img_trans
        self.mask_trans = mask_trans

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        
        img_t = self.img_trans(image)
        mask_t = self.mask_trans(mask).float()
        
        # Binary label: 1 = Pet, everything else = background
        binary_mask = (mask_t == 1.0).float()
        return img_t, binary_mask

split_idx = int(0.8 * len(valid_samples))
train_samples = valid_samples[:split_idx]
test_samples = valid_samples[split_idx:]

train_set = OxfordPetKaggleDataset(train_samples, img_transform, mask_transform)
test_set = OxfordPetKaggleDataset(test_samples, img_transform, mask_transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


---
## 🏗️ PART 2: REQUIREMENT 1 - BUILD A U-NET FROM SCRATCH


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=1):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = DoubleConv(256, 512)
        
        # Decoder with Skip Connections
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        
        # Final Conv 1x1
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        p1 = self.pool1(s1)
        
        s2 = self.enc2(p1)
        p2 = self.pool2(s2)
        
        s3 = self.enc3(p2)
        p3 = self.pool3(s3)
        
        b = self.bottleneck(p3)
        
        d3 = self.up3(b)
        d3 = torch.cat([d3, s3], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, s2], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, s1], dim=1)
        d1 = self.dec1(d1)
        
        return self.final_conv(d1)

model_unet = UNet(in_channels=3, num_classes=1).to(device)
dummy = torch.randn(2, 3, 128, 128).to(device)
out = model_unet(dummy)
print("Input shape  :", dummy.shape)
print("Output shape :", out.shape)
print("Total params :", sum(p.numel() for p in model_unet.parameters()))


---
## 📏 PART 3: LOSS FUNCTION AND EVALUATION METRICS (PIXEL ACC, IoU, DICE)


In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred).view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return 1.0 - dice

def compute_metrics(pred_mask, true_mask, smooth=1e-6):
    p_flat = pred_mask.view(-1).cpu().numpy()
    t_flat = true_mask.view(-1).cpu().numpy()
    
    pixel_acc = (p_flat == t_flat).mean() * 100
    intersection = np.logical_and(p_flat == 1, t_flat == 1).sum()
    union = np.logical_or(p_flat == 1, t_flat == 1).sum()
    
    iou = (intersection + smooth) / (union + smooth) * 100
    dice = (2.0 * intersection + smooth) / (p_flat.sum() + t_flat.sum() + smooth) * 100
    return pixel_acc, iou, dice


---
## 🚀 PART 4: TRAINING THE U-NET MODEL


In [ ]:
bce_criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_unet.parameters(), lr=1e-3)

EPOCHS = 5
history = {'train_loss': [], 'val_loss': [], 'pixel_acc': [], 'iou': [], 'dice': []}

print("=== STARTING U-NET TRAINING ===")
t_start = time.time()

for epoch in range(EPOCHS):
    model_unet.train()
    running_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model_unet(images)
        loss = bce_criterion(outputs, masks) + dice_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        
    epoch_train_loss = running_loss / max(1, len(train_set))
    
    model_unet.eval()
    val_loss = 0.0
    total_acc, total_iou, total_dice = 0.0, 0.0, 0.0
    with torch.no_grad():
        for images, masks in test_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model_unet(images)
            val_loss += (bce_criterion(outputs, masks) + dice_loss(outputs, masks)).item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            acc, iou, dice = compute_metrics(preds, masks)
            total_acc += acc
            total_iou += iou
            total_dice += dice
            
    nb_test = max(1, len(test_loader))
    epoch_val_loss = val_loss / max(1, len(test_set))
    epoch_acc = total_acc / nb_test
    epoch_iou = total_iou / nb_test
    epoch_dice = total_dice / nb_test
    
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['pixel_acc'].append(epoch_acc)
    history['iou'].append(epoch_iou)
    history['dice'].append(epoch_dice)
    
    print(f"Epoch [{epoch+1:2d}/{EPOCHS}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Pixel Acc: {epoch_acc:.2f}% | IoU: {epoch_iou:.2f}% | Dice: {epoch_dice:.2f}%")

print(f"Training finished in {(time.time()-t_start)/60:.2f} minutes!")


---
## 📊 PART 5: COMPREHENSIVE ANALYSIS CHARTS FOR SEMANTIC SEGMENTATION


In [ ]:
# Collect evaluation data on the test set to plot statistical distributions
test_ious = []
test_dices = []
test_pet_pixels = 0
test_bg_pixels = 0

model_unet.eval()
with torch.no_grad():
    for imgs, msks in test_loader:
        out = model_unet(imgs.to(device))
        bin_preds = (torch.sigmoid(out) > 0.5).float()
        for i in range(len(imgs)):
            _, iou_s, dice_s = compute_metrics(bin_preds[i], msks[i])
            test_ious.append(iou_s)
            test_dices.append(dice_s)
            test_pet_pixels += int(msks[i].sum().item())
            test_bg_pixels += int((1 - msks[i]).sum().item())
        if len(test_ious) >= 50:
            break

epochs_arr = range(1, EPOCHS + 1)
fig = plt.figure(figsize=(16, 12))

# 1. Loss curve
ax1 = fig.add_subplot(2, 3, 1)
ax1.plot(epochs_arr, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_arr, history['val_loss'], 'r--s', label='Val Loss')
ax1.set_title('1. Loss Curve', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

# 2. Pixel Accuracy curve
ax2 = fig.add_subplot(2, 3, 2)
ax2.plot(epochs_arr, history['pixel_acc'], 'g-^', linewidth=2)
ax2.set_title('2. Pixel Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, linestyle='--', alpha=0.6)

# 3. Compare IoU & Dice Score curves across epochs
ax3 = fig.add_subplot(2, 3, 3)
ax3.plot(epochs_arr, history['iou'], 'm-o', label='Mean IoU')
ax3.plot(epochs_arr, history['dice'], 'c-s', label='Dice Score')
ax3.set_title('3. IoU & Dice Score Progress', fontweight='bold')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Score (%)')
ax3.legend()
ax3.grid(True, linestyle='--', alpha=0.6)

# 4. Pie chart of Pet vs Background area (class imbalance)
ax4 = fig.add_subplot(2, 3, 4)
total_pix = test_pet_pixels + test_bg_pixels
if total_pix > 0:
    ax4.pie([test_pet_pixels, test_bg_pixels], labels=['Pet', 'Background'],
            colors=['#ff9999', '#66b3ff'], autopct='%1.1f%%', startangle=140, explode=(0.08, 0))
ax4.set_title('4. Pet vs Background Area Ratio', fontweight='bold')

# 5. Scatter plot: correlation between IoU and Dice Score
ax5 = fig.add_subplot(2, 3, 5)
if len(test_ious) > 0:
    ax5.scatter(test_ious, test_dices, c='darkblue', alpha=0.6, edgecolors='none')
    # Plot the theoretical curve Dice = 2*IoU / (1 + IoU)
    x_theory = np.linspace(1, 100, 100)
    y_theory = (2.0 * x_theory / (100.0 + x_theory)) * 100.0
    ax5.plot(x_theory, y_theory, 'r--', label='Theoretical curve')
    ax5.legend()
ax5.set_title('5. IoU vs Dice Score Correlation', fontweight='bold')
ax5.set_xlabel('IoU (%)')
ax5.set_ylabel('Dice Score (%)')
ax5.grid(True, linestyle='--', alpha=0.6)

# 6. Histogram of Dice Score distribution on the test set
ax6 = fig.add_subplot(2, 3, 6)
if len(test_dices) > 0:
    ax6.hist(test_dices, bins=12, color='orange', alpha=0.7, edgecolor='black')
ax6.set_title('6. Dice Score Distribution on Test Samples', fontweight='bold')
ax6.set_xlabel('Dice Score (%)')
ax6.set_ylabel('Number of images')
ax6.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


---
## 🖼️ PART 6: REQUIREMENT 2 - 4-COLUMN VISUALIZATION:
### *Input image ── Ground-truth mask ── Predicted mask ── Overlay Blended Mask*


In [ ]:
model_unet.eval()
test_iter = iter(test_loader)
images, masks = next(test_iter)

with torch.no_grad():
    preds = torch.sigmoid(model_unet(images.to(device))).cpu()
    preds_binary = (preds > 0.5).float()

num_samples = 4
fig, axes = plt.subplots(num_samples, 4, figsize=(15, 3.5 * num_samples))

for i in range(num_samples):
    # Column 1: Input Image
    img = images[i].numpy().transpose(1, 2, 0)
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Sample #{i+1}: Input Image", fontsize=11, fontweight='bold')
    axes[i, 0].axis('off')
    
    # Column 2: Ground-truth
    gt = masks[i].squeeze().numpy()
    axes[i, 1].imshow(gt, cmap='Blues')
    axes[i, 1].set_title("Ground-truth Mask", fontsize=11, fontweight='bold')
    axes[i, 1].axis('off')
    
    # Column 3: Predicted Mask
    pr = preds_binary[i].squeeze().numpy()
    acc_s, iou_s, dice_s = compute_metrics(preds_binary[i], masks[i])
    axes[i, 2].imshow(pr, cmap='Blues')
    axes[i, 2].set_title(f"Predicted (IoU: {iou_s:.1f}%)", fontsize=11, fontweight='bold')
    axes[i, 2].axis('off')
    
    # Column 4: Overlay Blended Mask (color overlay on the original image)
    overlay = img.copy()
    # Apply a semi-transparent red over the predicted region
    overlay[pr == 1] = overlay[pr == 1] * 0.5 + np.array([1.0, 0.0, 0.0]) * 0.5
    axes[i, 3].imshow(overlay)
    axes[i, 3].set_title("Overlay (Segmented Region)", fontsize=11, fontweight='bold')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()


---
## 🔬 PART 7: REQUIREMENT 3 - U-NET ARCHITECTURE ANALYSIS
1. **Encoder:** Extracts high-level semantic features (what the object is) while compressing spatial dimensions.
2. **Decoder:** Restores spatial resolution back to the original size via upsampling.
3. **Skip Connections:** Pass fine-grained spatial information directly from the Encoder to the Decoder to recover sharp boundaries.


---
## 🎯 PART 8: FREQUENTLY ASKED QUESTIONS (CHEAT-SHEET Q&A)
- **❓ Q1: Why does U-Net need Skip Connections?**  
  *Answer:* Max pooling in the Encoder loses precise spatial location information. Skip Connections pass high-resolution features directly to the Decoder so it can redraw accurate boundaries.
- **❓ Q2: Why combine BCE Loss and Dice Loss?**  
  *Answer:* BCE optimizes each pixel's probability independently, while Dice Loss optimizes the overlap area, which helps handle the class imbalance between object and background.
